# SHAP Analysis

Explain model predictions using SHAP TreeExplainer.

In [ ]:
import pickle
import numpy as np
import shap
import matplotlib.pyplot as plt
from pathlib import Path

# Load model
model_path = Path('../backend/ml/artifacts/model.pkl')
with open(model_path, 'rb') as f:
    model = pickle.load(f)

print(f'Model type: {type(model).__name__}')

In [ ]:
# Load training data
import sys
sys.path.insert(0, '../backend')
from ml.train import load_data, create_splits

df = load_data('../backend/data/output')
train, val, test = create_splits(df)

feature_cols = ['total_orders', 'total_refunds', 'total_amount', 'avg_amount',
                'max_amount', 'refund_rate', 'refund_ratio', 'high_amount']

X_train = train[feature_cols].values
print(f'Training samples: {len(X_train)}')

## SHAP Summary Plot (Beeswarm)

In [ ]:
# Compute SHAP values
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_train)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_train, feature_names=feature_cols, show=False)
plt.title('SHAP Summary Plot', fontsize=14)
plt.tight_layout()
plt.savefig('04_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## SHAP Feature Importance (Bar Plot)

In [ ]:
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_train, feature_names=feature_cols, plot_type='bar', show=False)
plt.title('SHAP Feature Importance', fontsize=14)
plt.tight_layout()
plt.savefig('04_shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## Individual Prediction Explanation

In [ ]:
# Explain a single prediction (first abuse case)
abuse_idx = np.where(train['label'].values == 1)[0]
if len(abuse_idx) > 0:
    idx = abuse_idx[0]
    print(f'Explaining prediction for account index {idx}')
    print(f'Actual label: Abuse')
    print(f'Predicted probability: {model.predict_proba(X_train[idx:idx+1])[0, 1]:.4f}')
    
    shap.force_plot(
        explainer.expected_value,
        shap_values[idx],
        X_train[idx],
        feature_names=feature_cols,
        matplotlib=True
    )
    plt.savefig('04_shap_force_plot.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No abuse cases found in training data')